# Transformer from Scratch: "Attention is All You Need"

Complete implementation of the Transformer architecture (Vaswani et al., 2017).

**What we'll build:**
1. Multi-head self-attention
2. Positional encoding
3. Encoder stack (6 layers)
4. Decoder stack (6 layers)
5. Complete Transformer for translation
6. Training with teacher forcing
7. Inference with greedy decoding

**Dataset:** Multi30k (English → German translation)

## 1. Imports and Setup

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import math
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from collections import Counter
import time

# Set random seed
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 2. Load Multi30k Dataset

In [ ]:
# Load dataset
dataset = load_dataset('bentrevett/multi30k')

# Use subset for faster training
TRAIN_SIZE = 5000
VAL_SIZE = 500

train_data = dataset['train'].select(range(TRAIN_SIZE))
val_data = dataset['validation'].select(range(VAL_SIZE))

print(f'Training samples: {len(train_data)}')
print(f'Validation samples: {len(val_data)}')
print()
print('Example:')
print(f"  EN: {train_data[200]['en']}")
print(f"  DE: {train_data[200]['de']}")

## 3. Tokenization & Vocabulary

In [ ]:
def tokenize(text):
    return text.lower().strip().split()

class Vocabulary:
    def __init__(self, name, min_freq=2):
        self.name = name
        self.min_freq = min_freq
        self.word2idx = {'<PAD>': 0, '<SOS>': 1, '<EOS>': 2, '<UNK>': 3}
        self.idx2word = {0: '<PAD>', 1: '<SOS>', 2: '<EOS>', 3: '<UNK>'}
        self.n_words = 4
    
    def build_vocab(self, sentences):
        counter = Counter()
        for sent in sentences:
            counter.update(tokenize(sent))
        
        for word, freq in counter.items():
            if freq >= self.min_freq:
                self.word2idx[word] = self.n_words
                self.idx2word[self.n_words] = word
                self.n_words += 1
    
    def encode(self, sentence):
        tokens = tokenize(sentence)
        return [self.word2idx.get(w, self.word2idx['<UNK>']) for w in tokens]
    
    def decode(self, indices):
        return [self.idx2word[idx] for idx in indices if idx not in [0, 1, 2]]

# Build vocabularies
src_vocab = Vocabulary('english', min_freq=2)
trg_vocab = Vocabulary('german', min_freq=2)

src_vocab.build_vocab([ex['en'] for ex in train_data])
trg_vocab.build_vocab([ex['de'] for ex in train_data])

print(f'English vocabulary: {src_vocab.n_words} words')
print(f'German vocabulary: {trg_vocab.n_words} words')

## 4. Dataset & DataLoader

In [ ]:
class TranslationDataset(torch.utils.data.Dataset):
    def __init__(self, data, src_vocab, trg_vocab):
        self.data = data
        self.src_vocab = src_vocab
        self.trg_vocab = trg_vocab
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        src = self.src_vocab.encode(self.data[idx]['en']) + [self.src_vocab.word2idx['<EOS>']]
        trg = [self.trg_vocab.word2idx['<SOS>']] + self.trg_vocab.encode(self.data[idx]['de']) + [self.trg_vocab.word2idx['<EOS>']]
        return torch.tensor(src), torch.tensor(trg)

def collate_fn(batch):
    src_batch, trg_batch = zip(*batch)
    src_padded = nn.utils.rnn.pad_sequence(src_batch, padding_value=0, batch_first=True)
    trg_padded = nn.utils.rnn.pad_sequence(trg_batch, padding_value=0, batch_first=True)
    return src_padded, trg_padded

# Create datasets
train_dataset = TranslationDataset(train_data, src_vocab, trg_vocab)
val_dataset = TranslationDataset(val_data, src_vocab, trg_vocab)

# Create dataloaders
BATCH_SIZE = 32

train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn
)
val_loader = torch.utils.data.DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn
)

# Test
src, trg = next(iter(train_loader))
print(f'Source batch shape: {src.shape}  # [batch, src_len]')
print(f'Target batch shape: {trg.shape}  # [batch, trg_len]')
print(f'\nNote: Transformer uses batch_first=True (different from Bahdanau/Luong)')

## 5. Hyperparameters

In [ ]:
# Model hyperparameters (smaller than paper for CPU/small GPU)
D_MODEL = 256          # Embedding dimension (paper: 512)
NUM_HEADS = 8          # Number of attention heads (paper: 8)
NUM_LAYERS = 3         # Encoder/decoder layers (paper: 6)
D_FF = 512             # Feed-forward dimension (paper: 2048)
MAX_SEQ_LEN = 100      # Maximum sequence length
DROPOUT = 0.1          # Dropout rate

# Training hyperparameters
LEARNING_RATE = 0.0001
N_EPOCHS = 15
CLIP = 1.0

print('Hyperparameters:')
print(f'  d_model: {D_MODEL}')
print(f'  num_heads: {NUM_HEADS}')
print(f'  num_layers: {NUM_LAYERS}')
print(f'  d_ff: {D_FF}')
print(f'  dropout: {DROPOUT}')
print(f'  epochs: {N_EPOCHS}')

## 6. Positional Encoding

In [ ]:
class PositionalEncoding(nn.Module):
    """Sinusoidal positional encoding from 'Attention is All You Need'"""
    
    def __init__(self, d_model, max_seq_len=5000, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        
        # Create positional encoding matrix
        pe = torch.zeros(max_seq_len, d_model)
        position = torch.arange(0, max_seq_len, dtype=torch.float).unsqueeze(1)
        
        # Compute the div_term for sinusoidal functions
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * 
                            -(math.log(10000.0) / d_model))
        
        # Apply sin to even indices, cos to odd indices
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        
        # Add batch dimension: [1, max_seq_len, d_model]
        pe = pe.unsqueeze(0)
        
        # Register as buffer (not a parameter)
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        """
        Args:
            x: [batch, seq_len, d_model]
        Returns:
            [batch, seq_len, d_model]
        """
        seq_len = x.size(1)
        x = x + self.pe[:, :seq_len]
        return self.dropout(x)

# Test
pe = PositionalEncoding(D_MODEL, MAX_SEQ_LEN, dropout=0.0)
x = torch.randn(2, 10, D_MODEL)
out = pe(x)
print(f'Input shape: {x.shape}')
print(f'Output shape: {out.shape}')

# Visualize positional encoding
plt.figure(figsize=(12, 6))
plt.imshow(pe.pe.squeeze(0)[:50].T.numpy(), cmap='RdBu', aspect='auto')
plt.colorbar()
plt.xlabel('Position')
plt.ylabel('Dimension')
plt.title('Positional Encoding Pattern')
plt.tight_layout()
plt.show()

## 7. Multi-Head Attention

The core building block of the Transformer.

In [ ]:
class MultiHeadAttention(nn.Module):
    """Multi-head attention mechanism"""
    
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        # Linear projections for Q, K, V
        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)
        
        # Output projection
        self.W_O = nn.Linear(d_model, d_model)
        
        self.dropout = nn.Dropout(dropout)
        self.scale = math.sqrt(self.d_k)
    
    def split_heads(self, x):
        """
        Split the last dimension into (num_heads, d_k)
        Args:
            x: [batch, seq_len, d_model]
        Returns:
            [batch, num_heads, seq_len, d_k]
        """
        batch_size, seq_len, d_model = x.size()
        x = x.view(batch_size, seq_len, self.num_heads, self.d_k)
        return x.transpose(1, 2)
    
    def combine_heads(self, x):
        """
        Combine heads back to original shape
        Args:
            x: [batch, num_heads, seq_len, d_k]
        Returns:
            [batch, seq_len, d_model]
        """
        batch_size, num_heads, seq_len, d_k = x.size()
        x = x.transpose(1, 2)
        return x.contiguous().view(batch_size, seq_len, self.d_model)
    
    def forward(self, Q, K, V, mask=None):
        """
        Args:
            Q: [batch, seq_len, d_model]
            K: [batch, seq_len, d_model]
            V: [batch, seq_len, d_model]
            mask: [batch, 1, 1, seq_len] or [batch, 1, seq_len, seq_len]
        Returns:
            output: [batch, seq_len, d_model]
            attention_weights: [batch, num_heads, seq_len, seq_len]
        """
        batch_size = Q.size(0)
        
        # 1. Linear projections
        Q = self.W_Q(Q)  # [batch, seq_len, d_model]
        K = self.W_K(K)
        V = self.W_V(V)
        
        # 2. Split into multiple heads
        Q = self.split_heads(Q)  # [batch, num_heads, seq_len, d_k]
        K = self.split_heads(K)
        V = self.split_heads(V)
        
        # 3. Scaled dot-product attention
        scores = torch.matmul(Q, K.transpose(-2, -1)) / self.scale
        # scores: [batch, num_heads, seq_len, seq_len]
        
        # 4. Apply mask (if provided)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        
        # 5. Softmax
        attention_weights = F.softmax(scores, dim=-1)
        attention_weights = self.dropout(attention_weights)
        
        # 6. Weighted sum of values
        output = torch.matmul(attention_weights, V)
        # output: [batch, num_heads, seq_len, d_k]
        
        # 7. Combine heads
        output = self.combine_heads(output)  # [batch, seq_len, d_model]
        
        # 8. Final linear projection
        output = self.W_O(output)
        
        return output, attention_weights

# Test
mha = MultiHeadAttention(D_MODEL, NUM_HEADS, DROPOUT)
x = torch.randn(2, 10, D_MODEL)
out, attn = mha(x, x, x)
print(f'Input shape: {x.shape}')
print(f'Output shape: {out.shape}')
print(f'Attention weights shape: {attn.shape}')
print(f'Parameters: {sum(p.numel() for p in mha.parameters()):,}')

## 8. Feed-Forward Network

In [ ]:
class FeedForward(nn.Module):
    """Position-wise feed-forward network"""
    
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.dropout = nn.Dropout(dropout)
        self.linear2 = nn.Linear(d_ff, d_model)
    
    def forward(self, x):
        # x: [batch, seq_len, d_model]
        x = self.linear1(x)  # [batch, seq_len, d_ff]
        x = F.relu(x)
        x = self.dropout(x)
        x = self.linear2(x)  # [batch, seq_len, d_model]
        return x

# Test
ffn = FeedForward(D_MODEL, D_FF, DROPOUT)
x = torch.randn(2, 10, D_MODEL)
out = ffn(x)
print(f'Input shape: {x.shape}')
print(f'Output shape: {out.shape}')

## 9. Encoder Layer

In [ ]:
class EncoderLayer(nn.Module):
    """Single encoder layer: Self-Attention + FFN"""
    
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        
        # Multi-head self-attention
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        
        # Feed-forward network
        self.ffn = FeedForward(d_model, d_ff, dropout)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout2 = nn.Dropout(dropout)
    
    def forward(self, x, mask=None):
        """
        Args:
            x: [batch, seq_len, d_model]
            mask: [batch, 1, 1, seq_len]
        Returns:
            [batch, seq_len, d_model]
        """
        # Self-attention with residual connection and layer norm
        attn_output, _ = self.self_attn(x, x, x, mask)
        x = self.norm1(x + self.dropout1(attn_output))
        
        # Feed-forward with residual connection and layer norm
        ffn_output = self.ffn(x)
        x = self.norm2(x + self.dropout2(ffn_output))
        
        return x

# Test
enc_layer = EncoderLayer(D_MODEL, NUM_HEADS, D_FF, DROPOUT)
x = torch.randn(2, 10, D_MODEL)
out = enc_layer(x)
print(f'Input shape: {x.shape}')
print(f'Output shape: {out.shape}')

## 10. Decoder Layer

In [ ]:
class DecoderLayer(nn.Module):
    """Single decoder layer: Masked Self-Attention + Cross-Attention + FFN"""
    
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        
        # Masked self-attention
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        
        # Cross-attention to encoder
        self.cross_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout2 = nn.Dropout(dropout)
        
        # Feed-forward network
        self.ffn = FeedForward(d_model, d_ff, dropout)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout3 = nn.Dropout(dropout)
    
    def forward(self, x, encoder_output, src_mask=None, tgt_mask=None):
        """
        Args:
            x: [batch, tgt_len, d_model] - decoder input
            encoder_output: [batch, src_len, d_model] - encoder output
            src_mask: [batch, 1, 1, src_len] - encoder padding mask
            tgt_mask: [batch, 1, tgt_len, tgt_len] - decoder causal + padding mask
        Returns:
            [batch, tgt_len, d_model]
        """
        # Masked self-attention (causal)
        self_attn_output, _ = self.self_attn(x, x, x, tgt_mask)
        x = self.norm1(x + self.dropout1(self_attn_output))
        
        # Cross-attention to encoder
        cross_attn_output, _ = self.cross_attn(
            x,                # Q from decoder
            encoder_output,   # K from encoder
            encoder_output,   # V from encoder
            src_mask
        )
        x = self.norm2(x + self.dropout2(cross_attn_output))
        
        # Feed-forward
        ffn_output = self.ffn(x)
        x = self.norm3(x + self.dropout3(ffn_output))
        
        return x

# Test
dec_layer = DecoderLayer(D_MODEL, NUM_HEADS, D_FF, DROPOUT)
x = torch.randn(2, 8, D_MODEL)
enc_out = torch.randn(2, 10, D_MODEL)
out = dec_layer(x, enc_out)
print(f'Decoder input shape: {x.shape}')
print(f'Encoder output shape: {enc_out.shape}')
print(f'Decoder output shape: {out.shape}')

## 11. Complete Transformer

In [ ]:
class Transformer(nn.Module):
    """Complete Transformer model for sequence-to-sequence tasks"""
    
    def __init__(
        self,
        src_vocab_size,
        tgt_vocab_size,
        d_model=512,
        num_heads=8,
        num_encoder_layers=6,
        num_decoder_layers=6,
        d_ff=2048,
        max_seq_len=5000,
        dropout=0.1,
        pad_idx=0
    ):
        super().__init__()
        
        self.d_model = d_model
        self.pad_idx = pad_idx
        
        # Embeddings
        self.src_embedding = nn.Embedding(src_vocab_size, d_model, padding_idx=pad_idx)
        self.tgt_embedding = nn.Embedding(tgt_vocab_size, d_model, padding_idx=pad_idx)
        self.pos_encoding = PositionalEncoding(d_model, max_seq_len, dropout)
        
        # Encoder stack
        self.encoder_layers = nn.ModuleList([
            EncoderLayer(d_model, num_heads, d_ff, dropout)
            for _ in range(num_encoder_layers)
        ])
        
        # Decoder stack
        self.decoder_layers = nn.ModuleList([
            DecoderLayer(d_model, num_heads, d_ff, dropout)
            for _ in range(num_decoder_layers)
        ])
        
        # Output projection
        self.fc_out = nn.Linear(d_model, tgt_vocab_size)
        
        self.dropout = nn.Dropout(dropout)
        
        # Initialize parameters
        self._init_parameters()
    
    def _init_parameters(self):
        """Initialize parameters with Xavier uniform"""
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)
    
    def create_padding_mask(self, seq):
        """
        Create mask for padding tokens
        Args:
            seq: [batch, seq_len]
        Returns:
            [batch, 1, 1, seq_len]
        """
        mask = (seq != self.pad_idx).unsqueeze(1).unsqueeze(2)
        return mask
    
    def create_causal_mask(self, size):
        """
        Create causal (lower triangular) mask
        Args:
            size: sequence length
        Returns:
            [1, 1, size, size]
        """
        mask = torch.tril(torch.ones(size, size)).unsqueeze(0).unsqueeze(0)
        return mask
    
    def encode(self, src, src_mask=None):
        """
        Encode source sequence
        Args:
            src: [batch, src_len]
            src_mask: [batch, 1, 1, src_len]
        Returns:
            [batch, src_len, d_model]
        """
        # Embed and add positional encoding
        x = self.src_embedding(src) * math.sqrt(self.d_model)
        x = self.pos_encoding(x)
        
        # Pass through encoder layers
        for layer in self.encoder_layers:
            x = layer(x, src_mask)
        
        return x
    
    def decode(self, tgt, encoder_output, src_mask=None, tgt_mask=None):
        """
        Decode target sequence
        Args:
            tgt: [batch, tgt_len]
            encoder_output: [batch, src_len, d_model]
            src_mask: [batch, 1, 1, src_len]
            tgt_mask: [batch, 1, tgt_len, tgt_len]
        Returns:
            [batch, tgt_len, d_model]
        """
        # Embed and add positional encoding
        x = self.tgt_embedding(tgt) * math.sqrt(self.d_model)
        x = self.pos_encoding(x)
        
        # Pass through decoder layers
        for layer in self.decoder_layers:
            x = layer(x, encoder_output, src_mask, tgt_mask)
        
        return x
    
    def forward(self, src, tgt):
        """
        Forward pass
        Args:
            src: [batch, src_len]
            tgt: [batch, tgt_len]
        Returns:
            [batch, tgt_len, tgt_vocab_size]
        """
        # Create masks
        src_mask = self.create_padding_mask(src).to(src.device)
        tgt_padding_mask = self.create_padding_mask(tgt).to(tgt.device)
        tgt_causal_mask = self.create_causal_mask(tgt.size(1)).to(tgt.device)
        tgt_mask = tgt_padding_mask & tgt_causal_mask
        
        # Encode and decode
        encoder_output = self.encode(src, src_mask)
        decoder_output = self.decode(tgt, encoder_output, src_mask, tgt_mask)
        
        # Project to vocabulary
        output = self.fc_out(decoder_output)
        
        return output

# Initialize model
model = Transformer(
    src_vocab_size=src_vocab.n_words,
    tgt_vocab_size=trg_vocab.n_words,
    d_model=D_MODEL,
    num_heads=NUM_HEADS,
    num_encoder_layers=NUM_LAYERS,
    num_decoder_layers=NUM_LAYERS,
    d_ff=D_FF,
    max_seq_len=MAX_SEQ_LEN,
    dropout=DROPOUT,
    pad_idx=0
).to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model initialized with {total_params:,} trainable parameters')

# Test forward pass
src, trg = next(iter(train_loader))
src, trg = src.to(device), trg.to(device)
output = model(src, trg[:, :-1])  # Exclude last token from target
print(f'\nTest forward pass:')
print(f'  Source shape: {src.shape}')
print(f'  Target shape: {trg[:, :-1].shape}')
print(f'  Output shape: {output.shape}')

## 12. Training & Evaluation Functions

In [ ]:
def train_epoch(model, loader, optimizer, criterion, clip, device):
    model.train()
    epoch_loss = 0
    
    for src, trg in loader:
        src, trg = src.to(device), trg.to(device)
        
        optimizer.zero_grad()
        
        # Forward pass
        # Input: all tokens except last
        # Target: all tokens except first (<SOS>)
        output = model(src, trg[:, :-1])
        
        # Reshape for loss
        output_dim = output.shape[-1]
        output = output.contiguous().view(-1, output_dim)
        trg = trg[:, 1:].contiguous().view(-1)
        
        # Compute loss
        loss = criterion(output, trg)
        
        # Backward pass
        loss.backward()
        
        # Clip gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        
        optimizer.step()
        
        epoch_loss += loss.item()
    
    return epoch_loss / len(loader)


def evaluate(model, loader, criterion, device):
    model.eval()
    epoch_loss = 0
    
    with torch.no_grad():
        for src, trg in loader:
            src, trg = src.to(device), trg.to(device)
            
            # Forward pass
            output = model(src, trg[:, :-1])
            
            # Reshape for loss
            output_dim = output.shape[-1]
            output = output.contiguous().view(-1, output_dim)
            trg = trg[:, 1:].contiguous().view(-1)
            
            # Compute loss
            loss = criterion(output, trg)
            
            epoch_loss += loss.item()
    
    return epoch_loss / len(loader)

print('Training functions ready!')

## 13. Train the Model

In [ ]:
# Setup training
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, betas=(0.9, 0.98), eps=1e-9)
criterion = nn.CrossEntropyLoss(ignore_index=0)  # Ignore padding

# Training loop
train_losses = []
val_losses = []
best_val_loss = float('inf')

print('Training started...\n')
start_time = time.time()

for epoch in range(N_EPOCHS):
    epoch_start = time.time()
    
    train_loss = train_epoch(model, train_loader, optimizer, criterion, CLIP, device)
    val_loss = evaluate(model, val_loader, criterion, device)
    
    epoch_time = time.time() - epoch_start
    
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    
    print(f'Epoch {epoch+1:02} | Time: {epoch_time:.0f}s | '
          f'Train Loss: {train_loss:.3f} | Val Loss: {val_loss:.3f}')
    
    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'transformer_best.pt')

total_time = time.time() - start_time
print(f'\nTraining complete! Total time: {total_time/60:.1f} minutes')
print(f'Best validation loss: {best_val_loss:.3f}')

## 14. Visualize Training

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(train_losses, label='Train Loss', linewidth=2)
plt.plot(val_losses, label='Val Loss', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Transformer Training Progress')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 15. Greedy Decoding for Inference

In [ ]:
def greedy_decode(model, src, src_vocab, trg_vocab, device, max_len=50):
    """
    Greedy decoding: always pick the highest probability token
    """
    model.eval()
    
    # Tokenize and encode source
    src_tokens = src_vocab.encode(src) + [src_vocab.word2idx['<EOS>']]
    src_tensor = torch.tensor(src_tokens).unsqueeze(0).to(device)  # [1, src_len]
    
    # Create source mask
    src_mask = model.create_padding_mask(src_tensor).to(device)
    
    with torch.no_grad():
        # Encode source
        encoder_output = model.encode(src_tensor, src_mask)
        
        # Start with <SOS>
        ys = torch.ones(1, 1).fill_(trg_vocab.word2idx['<SOS>']).long().to(device)
        
        for i in range(max_len - 1):
            # Create target masks
            tgt_padding_mask = model.create_padding_mask(ys).to(device)
            tgt_causal_mask = model.create_causal_mask(ys.size(1)).to(device)
            tgt_mask = tgt_padding_mask & tgt_causal_mask
            
            # Decode
            decoder_output = model.decode(ys, encoder_output, src_mask, tgt_mask)
            
            # Get next token
            prob = model.fc_out(decoder_output[:, -1])
            next_word = torch.argmax(prob, dim=-1)
            
            # Append to output
            ys = torch.cat([ys, next_word.unsqueeze(0)], dim=1)
            
            # Stop if <EOS>
            if next_word.item() == trg_vocab.word2idx['<EOS>']:
                break
    
    # Decode translation
    tokens = ys.squeeze(0).tolist()
    translation = trg_vocab.decode(tokens)
    
    return translation

# Test translation
print('Translation Examples:\n')
for i in [0, 10, 50, 100]:
    src_sent = val_data[i]['en']
    trg_sent = val_data[i]['de']
    
    translation = greedy_decode(model, src_sent, src_vocab, trg_vocab, device)
    
    print(f'Source:      {src_sent}')
    print(f'Target:      {trg_sent}')
    print(f'Predicted:   {" ".join(translation)}')
    print()

## 16. Visualize Attention Weights

In [ ]:
def get_attention_maps(model, src, trg, src_vocab, trg_vocab, device):
    """
    Get attention weights from encoder and decoder
    """
    model.eval()
    
    # Tokenize
    src_tokens = src_vocab.encode(src) + [src_vocab.word2idx['<EOS>']]
    trg_tokens = [trg_vocab.word2idx['<SOS>']] + trg_vocab.encode(trg) + [trg_vocab.word2idx['<EOS>']]
    
    src_tensor = torch.tensor(src_tokens).unsqueeze(0).to(device)
    trg_tensor = torch.tensor(trg_tokens).unsqueeze(0).to(device)
    
    # Create masks
    src_mask = model.create_padding_mask(src_tensor).to(device)
    tgt_padding_mask = model.create_padding_mask(trg_tensor).to(device)
    tgt_causal_mask = model.create_causal_mask(trg_tensor.size(1)).to(device)
    tgt_mask = tgt_padding_mask & tgt_causal_mask
    
    with torch.no_grad():
        # Get encoder output
        x = model.src_embedding(src_tensor) * math.sqrt(model.d_model)
        x = model.pos_encoding(x)
        
        # Get encoder attention from last layer
        for i, layer in enumerate(model.encoder_layers):
            if i == len(model.encoder_layers) - 1:
                # Get attention weights from last encoder layer
                _, encoder_attn = layer.self_attn(x, x, x, src_mask)
            x = layer(x, src_mask)
        
        encoder_output = x
        
        # Get decoder output and cross-attention
        y = model.tgt_embedding(trg_tensor) * math.sqrt(model.d_model)
        y = model.pos_encoding(y)
        
        for i, layer in enumerate(model.decoder_layers):
            if i == len(model.decoder_layers) - 1:
                # Get attention from last decoder layer
                _, self_attn = layer.self_attn(y, y, y, tgt_mask)
                y = layer.norm1(y + layer.dropout1(layer.self_attn(y, y, y, tgt_mask)[0]))
                _, cross_attn = layer.cross_attn(y, encoder_output, encoder_output, src_mask)
            y = layer(y, encoder_output, src_mask, tgt_mask)
    
    return encoder_attn, self_attn, cross_attn

# Visualize attention for an example
example_idx = 10
src_sent = val_data[example_idx]['en']
trg_sent = val_data[example_idx]['de']

print(f'Source: {src_sent}')
print(f'Target: {trg_sent}\n')

encoder_attn, decoder_self_attn, decoder_cross_attn = get_attention_maps(
    model, src_sent, trg_sent, src_vocab, trg_vocab, device
)

# Get tokens for labels
src_tokens = src_vocab.encode(src_sent) + [src_vocab.word2idx['<EOS>']]
src_words = [src_vocab.idx2word[idx] for idx in src_tokens]
trg_tokens = [trg_vocab.word2idx['<SOS>']] + trg_vocab.encode(trg_sent) + [trg_vocab.word2idx['<EOS>']]
trg_words = [trg_vocab.idx2word[idx] for idx in trg_tokens]

# Plot encoder self-attention (first head)
plt.figure(figsize=(10, 8))
sns.heatmap(
    encoder_attn[0, 0].cpu().numpy(),
    xticklabels=src_words,
    yticklabels=src_words,
    cmap='Blues',
    cbar=True
)
plt.title('Encoder Self-Attention (Head 1, Last Layer)')
plt.xlabel('Key')
plt.ylabel('Query')
plt.tight_layout()
plt.show()

# Plot decoder cross-attention (first head)
plt.figure(figsize=(12, 8))
sns.heatmap(
    decoder_cross_attn[0, 0].cpu().numpy(),
    xticklabels=src_words,
    yticklabels=trg_words,
    cmap='Reds',
    cbar=True
)
plt.title('Decoder Cross-Attention (Head 1, Last Layer)\nShows which source words the decoder attends to')
plt.xlabel('Source (Key)')
plt.ylabel('Target (Query)')
plt.tight_layout()
plt.show()

## 17. Comparison with Previous Approaches

In [ ]:
print('='*80)
print('COMPARISON: Transformer vs Bahdanau vs Luong')
print('='*80)

comparison = """
┌────────────────────────────────────────────────────────────────────────────┐
│                         ARCHITECTURE COMPARISON                            │
├────────────────────────────────────────────────────────────────────────────┤
│                                                                            │
│ BAHDANAU (2015)                                                            │
│   - RNN/LSTM encoder + decoder                                             │
│   - Cross-attention (decoder → encoder)                                    │
│   - Sequential processing (slow)                                           │
│   - Attention: tanh(W_h·h_i + W_s·s_t-1)                                   │
│                                                                            │
│ LUONG (2015)                                                               │
│   - RNN/LSTM encoder + decoder                                             │
│   - Cross-attention (decoder → encoder)                                    │
│   - Sequential processing (slow)                                           │
│   - Attention: dot/general/concat scoring                                  │
│                                                                            │
│ TRANSFORMER (2017) ✨                                                       │
│   - NO RNN! Pure attention-based                                           │
│   - Self-attention (within sequence)                                       │
│   - Cross-attention (decoder → encoder)                                    │
│   - Fully parallel processing (fast!)                                      │
│   - Multi-head attention (8 heads)                                         │
│   - Positional encoding (explicit position info)                           │
│   - Attention: softmax(QK^T/√d_k)V                                         │
│                                                                            │
└────────────────────────────────────────────────────────────────────────────┘

┌────────────────────────────────────────────────────────────────────────────┐
│                          KEY INNOVATIONS                                   │
├────────────────────────────────────────────────────────────────────────────┤
│                                                                            │
│  1. Self-Attention                                                         │
│     Every position attends to all positions in same sequence               │
│     O(1) path between any two positions                                    │
│                                                                            │
│  2. Multi-Head Attention                                                   │
│     8 heads learn different relationship types                             │
│     Head 1: syntax, Head 2: semantics, etc.                                │
│                                                                            │
│  3. No Recurrence                                                          │
│     Fully parallelizable across sequence                                   │
│     GPU-friendly, much faster training                                     │
│                                                                            │
│  4. Positional Encoding                                                    │
│     PE(pos,2i) = sin(pos/10000^(2i/d))                                     │
│     PE(pos,2i+1) = cos(pos/10000^(2i/d))                                   │
│                                                                            │
│  5. Residual Connections + LayerNorm                                       │
│     Enable training very deep networks (6+ layers)                         │
│     Stable gradient flow                                                   │
│                                                                            │
└────────────────────────────────────────────────────────────────────────────┘

┌────────────────────────────────────────────────────────────────────────────┐
│                        PERFORMANCE COMPARISON                              │
├────────────────────────────────────────────────────────────────────────────┤
│                                                                            │
│  Metric              │ Bahdanau/Luong  │  Transformer                      │
│  ────────────────────┼─────────────────┼───────────────────────────────    │
│  Parallelization     │  Encoder only   │  Fully parallel                   │
│  Training speed      │  Slow           │  Fast                             │
│  Long-range deps     │  Weak (O(n))    │  Strong (O(1))                    │
│  Memory complexity   │  O(n)           │  O(n²)                            │
│  Translation quality │  Good           │  Better                           │
│  Interpretability    │  Medium         │  High (attention maps)            │
│                                                                            │
└────────────────────────────────────────────────────────────────────────────┘
"""

print(comparison)

print('\nKey Takeaway:')
print('  Transformer eliminates recurrence entirely, using only attention.')
print('  This makes it faster, more parallelizable, and better at capturing')
print('  long-range dependencies than RNN-based approaches.')

## 18. Summary

### What We Implemented:
1. **Multi-head self-attention** - The core mechanism
2. **Positional encoding** - Sinusoidal position embeddings
3. **Encoder** - Stack of self-attention + FFN layers
4. **Decoder** - Stack of masked self-attention + cross-attention + FFN
5. **Complete Transformer** - Full encoder-decoder architecture
6. **Training** - Teacher forcing with proper masking
7. **Inference** - Greedy decoding

### Architecture Summary:
```
Input → Embedding + Positional Encoding
     ↓
   ENCODER (N=3 layers)
     [Self-Attention + FFN] × 3
     ↓
   DECODER (N=3 layers)
     [Masked Self-Attention + Cross-Attention + FFN] × 3
     ↓
   Linear + Softmax → Output
```

### Key Components:
- **d_model**: 256 (embedding dimension)
- **num_heads**: 8 (attention heads)
- **num_layers**: 3 (encoder/decoder depth)
- **d_ff**: 512 (feed-forward hidden size)

### Innovations:
1. No recurrence - fully parallel
2. Self-attention - O(1) path between positions
3. Multi-head - learn multiple representations
4. Positional encoding - explicit position info

### Next Steps:
1. Try beam search for better translations
2. Implement learning rate scheduling (warmup + decay)
3. Add label smoothing
4. Train on larger dataset
5. Explore BERT (encoder-only) and GPT (decoder-only) variants
6. Implement efficient attention variants (sparse, linear)

**This architecture revolutionized NLP and led to BERT, GPT, T5, and all modern LLMs!**